In [1]:
import os
import sys

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import DeviceStatsMonitor
from pytorch_lightning.loggers import CSVLogger
from model import EventRespirationNet
from aedat_dataset import AEDATRespirationDataset
from datamodule import AEDATDataModule 
import matplotlib.pyplot as plt


In [2]:
def plot_loss_curves(train_losses, val_losses):
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss', marker='o')
    plt.plot(val_losses, label='Validation Loss', marker='x')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.show()
    plt.savefig("loss_curve.png")
    
    
def plot_predictions(y_true, y_pred, title="Predicted vs Ground Truth RR"):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, c='blue', alpha=0.6, label='Predictions')
    plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], 'r--', label='Ideal')
    plt.xlabel('Ground Truth RR')
    plt.ylabel('Predicted RR')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()
    
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    mode="min",
    verbose=True
)

class LossHistory(Callback):
    def __init__(self):
        self.train_losses = []
        self.val_losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        loss = trainer.callback_metrics.get("train_loss")
        if loss is not None:
            self.train_losses.append(loss.item())

    def on_validation_epoch_end(self, trainer, pl_module):
        loss = trainer.callback_metrics.get("val_loss")
        if loss is not None:
            self.val_losses.append(loss.item())


In [3]:
    # 1) Instantiate model
    model = EventRespirationNet(lr=1e-4)
    loss_history_cb = LossHistory()

    # 2) Instantiate DataModule pointing at your AEDAT folder and GT CSV
    data_module = AEDATDataModule(
        data_dir="../data/042025",
        csv_path="../data/ground_truth.csv",
        batch_size=8,
        frames_per_sample=200
    )

    # 3) Trainer
    trainer = Trainer(
        max_epochs=50,
        callbacks=[loss_history_cb, early_stop, DeviceStatsMonitor()],
        accelerator="auto",
        devices="auto",
        log_every_n_steps=10
    )

    # 4) Fit
    trainer.fit(model, datamodule=data_module)
    
    plot_loss_curves(loss_history_cb.train_losses, loss_history_cb.val_losses)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/conda/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precisi

Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Failed to read data, skipping.
Skipping patient_7_1,0m_Good_natural_light_reading_3.aedat4: No matching GT RR
Skipping patient_7_1,5m_Good_natural_light_reading_3.aedat4: No matching GT RR


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type       | Params | Mode 
-------------------------------------------------
0 | encoder   | Sequential | 250 K  | train
1 | regressor | Sequential | 8.3 K  | train
-------------------------------------------------
259 K     Trainable params
0         Non-trainable params
259 K     Total params
1.036     Total estimated model params size (MB)
30        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.49 GiB. GPU 0 has a total capacty of 79.25 GiB of which 588.06 MiB is free. Including non-PyTorch memory, this process has 5.97 GiB memory in use. Of the allocated memory 5.50 GiB is allocated by PyTorch, and 1024.00 KiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF